# Seasonal and synoptic predictability of daily solar irradiance across Brazil

Reproduces every table and figure of:

> Caneppele FL, Tieghi CP, Almeida CFM, Nogueira CEC, Siqueira JAC, Frigo MM, Malagueta DC, Godinho EZ.
> *Maritime exposure and latitude control different components of daily solar irradiance predictability across Brazil.*
> Submitted to Theoretical and Applied Climatology.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcaneppele/s3-solar-predictability-brazil/blob/main/notebooks/S3_pipeline.ipynb)

**What it does** (11 cities, NASA POWER daily data 2013–2023, all in one run):
1. six regressors (RF, GBT, SVM, ANN, MARS, CART) + climatology and Hargreaves–Samani references,
   random 80/20 split and year-by-year walk-forward validation (test years 2018–2023);
2. distance to the coast and climatological descriptors;
3. decomposition R² = f + SS(1 − f), Spearman/Mann–Whitney tests and bootstrap intervals;
4. Table 2, Online Resources 1–2 and Figs. 1–5.

Runtime: about 15–20 min on a standard Colab runtime (2 CPUs).

## 1. Setup
In Colab this clones the repository and compiles the vendored MARS implementation (py-earth, patched for current NumPy/scikit-learn; see `vendor/README_py-earth.md`). Locally, run it from the repository root after `pip install -r requirements.txt`.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("s3-solar-predictability-brazil"):
        subprocess.run(["git", "clone", "-q", "https://github.com/fcaneppele/s3-solar-predictability-brazil.git"], check=True)
    os.chdir("s3-solar-predictability-brazil")
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "global-land-mask", "cython<3", "setuptools<70", "wheel"], check=True)
sys.path.insert(0, os.path.abspath("vendor/py-earth"))
os.environ["PYTHONPATH"] = os.pathsep.join([os.path.abspath("vendor/py-earth"), os.path.abspath("src"), os.environ.get("PYTHONPATH", "")])
try:
    import pyearth
except ImportError:
    # compile the vendored MARS implementation in place (about 1 min)
    subprocess.run([sys.executable, "setup.py", "-q", "build_ext", "--inplace", "--cythonize"], cwd="vendor/py-earth", check=True)
    import importlib; importlib.invalidate_caches()
import pyearth, numpy, sklearn, pandas
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| scikit-learn", sklearn.__version__, "| pandas", pandas.__version__)
sys.path.insert(0, "src")
import s3pipe

## 2. Models and references (11 cities)

In [ ]:
import time; t0 = time.time()
metrics, krs = s3pipe.run_models(n_jobs=2)
print(f"done in {(time.time()-t0)/60:.1f} min;", len(metrics), "metric rows")

## 3. Geography and climatological descriptors

In [ ]:
s3pipe.coast_distance()
desc = s3pipe.descriptors()
desc.round(3)

## 4. Seasonal–synoptic decomposition and tests

In [ ]:
dec, med, tests = s3pipe.decomposition()
print("max |R2 - (f + SS(1-f))| =", dec.identity_err.abs().max())
tests[tests.design.isin(["walkforward", "random", "-", "walkforward_bootstrap"])].round(3)

## 5. Table 2 and Online Resources

In [ ]:
t2 = s3pipe.tables()
t2.round(2)

## 6. Figures 1–5 (saved to `results/figures/` as TIFF 600 dpi, EPS, PDF and PNG)

In [ ]:
s3pipe.figures()
from IPython.display import Image, display
for i in range(1, 6):
    display(Image(filename=f"results/figures/Fig{i}.png"))

## Notes on reproducibility
The archived results were produced with Python 3.10.x, NumPy 1.23.5, scikit-learn 1.2.2, pandas 2.3.3 and SciPy 1.10.1
(`requirements.txt`). Newer library versions (e.g. on Colab) can change the last decimals of the stochastic learners
(RF, ANN); the conclusions and all reported two-decimal values of the between-site statistics are unaffected.